In [56]:
from sklearn.datasets import make_classification
import torch

In [57]:
# Step 1: Create a synthetic classification dataset using sklearn
x, y  = make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

In [58]:
x

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [59]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [60]:
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [61]:
x

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [62]:
from torch.utils.data import Dataset, DataLoader

In [63]:
# Dataclass
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [64]:
dataset = CustomDataset(x, y)

In [65]:
len(dataset)

10

In [66]:
dataset[0]

(tensor([ 1.0683, -0.9701]), tensor(1))

In [67]:
# Dataloader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [68]:
for batch_features, batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("---"*50)

tensor([[-0.9382, -0.5430],
        [-2.8954,  1.9769]])
tensor([1, 0])
------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [-0.5872, -1.9717]])
tensor([1, 0])
------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1, 0])
------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-0.7206, -0.9606]])
tensor([0, 0])
------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [ 1.8997,  0.8344]])
tensor([1, 1])
------------------------------------

## Cancer problem in 4_NN_Module/pipeline.ipynb

In [69]:
import numpy as np
import pandas as pd
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [70]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [71]:
df.drop(['id', 'Unnamed: 32'], axis=1, inplace=True)

In [72]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [73]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [74]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [75]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()


In [76]:
# Dataclass
class CustomDataset2(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [ ]:
# Dataset
train_dataset = CustomDataset2(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset2(X_test_tensor, y_test_tensor)

In [78]:
train_dataset[10]

(tensor([ 0.2635,  1.3940,  0.1609,  0.1419, -0.1927, -1.0060, -0.8195, -0.5189,
         -0.9283, -1.1542,  2.8479,  1.6853,  2.4979,  1.3918, -0.0712, -0.8122,
         -0.4667,  1.1372, -1.5458, -0.7706, -0.2470, -0.0716, -0.3360, -0.3038,
         -1.6675, -1.2640, -1.1790, -1.3067, -2.2051, -1.5862]),
 tensor(1.))

In [79]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [80]:
class MySimpleNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)

        return out


In [81]:
learning_rate = 0.1
epochs = 25

In [82]:
model = MySimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
loss_function = nn.BCELoss()

## Training pipeline

In [83]:
#define loop
for epoch in range(epochs):
    
    for batch_features, batch_labels in train_loader:
        # Forward pass
        y_pred = model(batch_features)

        # Loss calculation
        loss = loss_function(y_pred, batch_labels.view(-1, 1))

        # Clear gradients
        optimizer.zero_grad()

        # Backward pass
        loss.backward()

        # Parameter update
        optimizer.step()

        # Print loss in each epoch
        print(f'Epoch: {epoch + 1}, loss: {loss.item()}')

Epoch: 1, loss: 0.633176863193512
Epoch: 1, loss: 0.44251030683517456
Epoch: 1, loss: 0.47776180505752563
Epoch: 1, loss: 0.31899943947792053
Epoch: 1, loss: 0.26113903522491455
Epoch: 1, loss: 0.23927335441112518
Epoch: 1, loss: 0.24699069559574127
Epoch: 1, loss: 0.28629082441329956
Epoch: 1, loss: 0.3054821789264679
Epoch: 1, loss: 0.18570548295974731
Epoch: 1, loss: 0.22892038524150848
Epoch: 1, loss: 0.22670386731624603
Epoch: 1, loss: 0.2539885342121124
Epoch: 1, loss: 0.15807466208934784
Epoch: 1, loss: 0.1574523150920868
Epoch: 2, loss: 0.14377684891223907
Epoch: 2, loss: 0.11690005660057068
Epoch: 2, loss: 0.19141584634780884
Epoch: 2, loss: 0.15207885205745697
Epoch: 2, loss: 0.2413950115442276
Epoch: 2, loss: 0.1948709338903427
Epoch: 2, loss: 0.15312403440475464
Epoch: 2, loss: 0.21336007118225098
Epoch: 2, loss: 0.18442033231258392
Epoch: 2, loss: 0.14180085062980652
Epoch: 2, loss: 0.14462967216968536
Epoch: 2, loss: 0.1675446778535843
Epoch: 2, loss: 0.1598966270685196
E

In [84]:
model.eval()
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()

        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Overall Accuracy: {overall_accuracy:.4f}')

Overall Accuracy: 0.9349
